# 04 — Interface robustness and evidence audit

Permitted inputs: mathematical fixtures from frozen 03 only. No participant data, clinical claims or cross-dataset validation. Outputs: 38 Brier/interval sweep rows, nine measured CPU timing summaries, endpoint/failure contracts, two figure families, and a bounded claim ledger. The protocol was committed before a successful result execution. Each perturbation is applied per repeat before probability averaging.

Figure contract: **Times New Roman, minimum 14-point native text, 1000 DPI PNG plus PDF/SVG**. Screen previews are 150 DPI, explicitly not submission rasters. A licensed project font is required; silent fallback is forbidden.

In [1]:
from pathlib import Path
import sys, json, subprocess
from IPython.display import Markdown, display
ROOT=Path.cwd()
if not (ROOT/'pyproject.toml').exists(): ROOT=ROOT.parent
assert (ROOT/'pyproject.toml').exists()
sys.path.insert(0,str(ROOT/'scripts'))
from s01_robustness_audit import load_protocol, run_audit, write_audit, check_contracts
protocol=load_protocol()
display(protocol)

{'schema_version': '1.0',
 'protocol_id': 's01-interface-robustness-v1',
 'scope': 'controlled_mathematical_interface_checks_not_signal_or_clinical_robustness',
 'base_protocol': 's01-synthetic-comparator-v1',
 'fixture_seeds': [20261011, 20261012, 20261013],
 'base_quality': 'informative',
 'repeat_aggregation': "apply every perturbation to each repeat's held-out predictions and training-derived weights first, then average fused probabilities; never multiply separately averaged probabilities and weights",
 'quality_ablation': ['informative',
  'permuted_record_weights',
  'reversed_weights',
  'equal_weights'],
 'availability': ['fmri_missing', 'eeg_missing'],
 'missing_fractions': [0.0, 0.1, 0.25, 0.5, 0.75, 1.0],
 'missingness_seed': 20261014,
 'missingness': 'independent uniform variate per generated record; nested masks across requested fractions; actual realized counts reported; same masks for all methods',
 'availability_rule': 'one unavailable modality receives zero weight in B

## Frozen perturbations and measured public-kernel timing

Availability masks are shared across methods. Both remove unavailable modalities; both-unavailable and zero reliability reject. Probability shrinkage is an interface transformation, not real signal noise. Record-bootstrap intervals are conditional on fitted models; no refitting, population inference, or multiple-comparison significance claim. CPU timings are medians/IQR over 15 measured calls after three warmups, not a full pipeline or peak-memory audit.

In [2]:
audit=run_audit(protocol)
path=write_audit(audit)
assert len(audit['sweeps'])==38 and len(audit['timings'])==9
assert audit['base_matches_03'] and audit['deterministic_sweep_replay']
display(audit['contracts'],audit['timings'])

{'only_eeg_endpoint': True,
 'only_fmri_endpoint': True,
 'failure_cases': [{'case': 'both_unavailable',
   'exception': 'ValueError',
   'diagnostic': 'at least one modality must have positive quality'},
  {'case': 'invalid_availability',
   'exception': 'ValueError',
   'diagnostic': 'availability must be binary'},
  {'case': 'nonfinite_weight',
   'exception': 'ValueError',
   'diagnostic': 'weight must be finite in [0,1]'},
  {'case': 'unavailable_positive_mass',
   'exception': 'ValueError',
   'diagnostic': 'at least one modality must have positive quality'}],
 'zero_mass_limitation': 'Availability alone cannot rescue a modality with exactly zero reliability mass; explicit normalization rejects rather than inventing a weight.'}

[{'operation': 'soft_fusion',
  'records': 1000,
  'repetitions': 15,
  'median_ms': 0.0255,
  'q25_ms': 0.025349999999999998,
  'q75_ms': 0.0258,
  'min_ms': 0.0251,
  'max_ms': 0.0371},
 {'operation': 'quality_normalization',
  'records': 1000,
  'repetitions': 15,
  'median_ms': 0.0301,
  'q25_ms': 0.02785,
  'q75_ms': 0.0307,
  'min_ms': 0.0273,
  'max_ms': 0.0632},
 {'operation': 'held_out_utility',
  'records': 1000,
  'repetitions': 15,
  'median_ms': 0.027,
  'q25_ms': 0.026750000000000003,
  'q75_ms': 0.0273,
  'min_ms': 0.0262,
  'max_ms': 0.0287},
 {'operation': 'soft_fusion',
  'records': 10000,
  'repetitions': 15,
  'median_ms': 0.0505,
  'q25_ms': 0.0504,
  'q75_ms': 0.05085,
  'min_ms': 0.0501,
  'max_ms': 0.0571},
 {'operation': 'quality_normalization',
  'records': 10000,
  'repetitions': 15,
  'median_ms': 0.0729,
  'q25_ms': 0.0726,
  'q75_ms': 0.07325000000000001,
  'min_ms': 0.0722,
  'max_ms': 0.0814},
 {'operation': 'held_out_utility',
  'records': 10000,
  'rep

Saved 04 audit: 38 sweep rows; 9 timing rows


## Publication figure exports

Read all captions in the [gallery](figures/README.md). Resize vector figures intentionally for the manuscript: native font size alone does not guarantee readability after shrinking a dense multi-panel figure.

In [3]:
from s01_evidence_figures import generate
figure_records=generate('04')
for item in figure_records:
    display(Markdown(f"### {item['id']}\n\n![Screen preview](figures/previews/{item['id']}.png)\n\n{item['caption']}"))
{'figure_sets':len(figure_records),'png_dpi':1000,'font':'Times New Roman'}

### F13_interface_robustness

![Screen preview](figures/previews/F13_interface_robustness.png)

Controlled prediction-interface audit, not EEG/fMRI signal corruption or external validation. Missingness uses shared nested random masks (actual counts in JSON); both methods remove unavailable modalities. Shrinkage also scales quality-aware fMRI reliability mass; no model retraining. Apply perturbation per repeat before averaging. Points/95% intervals use 1,000 seed/class-stratified record resamples conditional on fitted models; intervals are not paired-difference significance tests. Both-unavailable and zero-reliability cases reject explicitly.

### F14_cpu_kernel_timing

![Screen preview](figures/previews/F14_cpu_kernel_timing.png)

Measured public-API kernel calls only: 3 warmups, 15 repetitions, one BLAS/OpenMP thread; inputs prepared outside timed region. Bars are timing IQR, not confidence intervals. Runtime-specific environment and raw summary ranges are in JSON. Not a full-pipeline benchmark, GPU result, cross-method speed claim or peak-memory measurement.

{'figure_sets': 2, 'png_dpi': 1000, 'font': 'Times New Roman'}

Exported F13_interface_robustness: Times New Roman, 1000 dpi
Exported F14_cpu_kernel_timing: Times New Roman, 1000 dpi


## Bounded evidence and release checks

These checks validate current source/artifact hashes and the aggregate-only boundary, independently reconcile saved calculations, and export a failure matrix, aggregate CSV tables, a claim-evidence ledger, release-readiness JSON, and checksums. Observed hardening gaps remain explicit open requirements; generating a report is not submission approval. Clean-install CI, later dataset compatibility, clinical utility, and external validation remain unverified.

In [4]:
checks={}
for name,args in [('evidence',['scripts/verify_s01_evidence.py']),('provenance',['scripts/s01_provenance_tutorial.py','--verify']),('release_boundary',['scripts/validate_release.py'])]:
    completed=subprocess.run([sys.executable,*args],cwd=ROOT,check=True,text=True,capture_output=True)
    checks[name]=completed.stdout.strip()
ledger={'scope':'synthetic_software_evidence_only','checks':checks,'supports':['frozen comparator and robustness protocols','availability endpoint and fail-closed contracts','1000 DPI Times New Roman exports with source hashes','runtime-specific CPU kernel timings'],'does_not_support':['clinical utility','real signal-noise robustness','cross-dataset generalization','peak native memory','all-platform clean-install CI certification']}
from neurofusion_qa.provenance import canonical_json
(ROOT/'notebooks/artifacts/04_claim_ledger.json').write_text(canonical_json(ledger)+'\n',encoding='utf-8')
from s01_release_evidence import run_release_audit
release_audit=run_release_audit()
display(ledger)
display({'calculations':release_audit['calculations'],'failure_matrix':release_audit['failure_matrix'],'submission_ready':release_audit['submission_ready']})

{'scope': 'synthetic_software_evidence_only',
 'checks': {'evidence': 'Evidence verified: frozen protocols; 72 matched folds; 17 figure sets / 51 exports; Times New Roman / 1000 DPI; 180 metric rows / 144 paired differences.',
  'provenance': 'Manifest verified: 11 inputs, 2 outputs',
  'release_boundary': 'Release validation passed: aggregate-only, protected access count 0, no prohibited artifacts.'},
 'supports': ['frozen comparator and robustness protocols',
  'availability endpoint and fail-closed contracts',
  '1000 DPI Times New Roman exports with source hashes',
  'runtime-specific CPU kernel timings'],
 'does_not_support': ['clinical utility',
  'real signal-noise robustness',
  'cross-dataset generalization',
  'peak native memory',
  'all-platform clean-install CI certification']}

{'calculations': {'passed': True,
  'identities_checked': 474,
  'scope': 'aggregate consistency; metric oracle and bootstrap-pairing tests are separate',
  'uncertainty': 'conditional record bootstrap, not population or model-refitting uncertainty',
  'brier_log_loss': 'Brier uses original probabilities; log loss clips at 1e-6 as frozen in 03',
  'binary_metrics': 'training-selected thresholds and repeat majority votes, ties positive; not thresholding mean probabilities'},
 'failure_matrix': {'cases': 44, 'passed': 44, 'open_gaps': []},
 'submission_ready': False}

Verifying frozen evidence and publication exports
Evidence verified: frozen protocols; 72 matched folds; 17 figure sets / 51 exports; Times New Roman / 1000 DPI; 180 metric rows / 144 paired differences.
Checking provenance
Checking release_boundary
Cross-checking calculations and executing failure matrix
Release audit: 474 calculation checks; 44/44 case requirements passed; 105 files checksummed; submission_ready=False
